The work done on olaf-llm-eswc2024 has demonstrated the performance of components based on LLMs.

After adding a DeepSeekGenerator to enable OLAF to use DeepSeek as a language model, the choice of DeepSeek API is justified by:

Cost-effectiveness: DeepSeek offers a competitive pricing model, ideal for budget-constrained projects.

High performance: State-of-the-art models and low latency ensure fast and accurate responses.

Ease of integration: Clear documentation and simple SDKs accelerate development.

Reliability: A robust infrastructure and strong customer support ensure maximum availability.

Ethics: DeepSeek emphasizes transparency and allows adjustments to comply with ethical guidelines.

In summary, DeepSeek API provides an economical, high-performance, and reliable solution for OLAF's LLMComponent.

In [3]:
import os
from dotenv import load_dotenv

import openai
import spacy

from olaf import Pipeline
from olaf.commons.errors import MissingEnvironmentVariable
# from olaf.commons.llm_tools import LLMGenerator, DeepSeekGenerator
from olaf.commons.logging_config import logger
from olaf.commons.prompts import (
    deepseek_prompt_concept_term_extraction,
    deepseek_prompt_concept_extraction,
    deepseek_prompt_relation_extraction,
    deepseek_prompt_relation_term_extraction
)

from olaf.repository.serialiser import BaseOWLSerialiser
from olaf.pipeline.pipeline_component.concept_relation_extraction import(
    LLMBasedConceptExtraction,
    LLMBasedRelationExtraction
)
from olaf.pipeline.pipeline_component.term_extraction import LLMTermExtraction
from olaf.repository.corpus_loader import TextCorpusLoader
from olaf.repository.serialiser import KRJSONSerialiser

ImportError: cannot import name 'deepseek_prompt_concept_term_extraction' from 'olaf.commons.prompts' (/home/talibe/Bureau/Stage Insa/OLAF Research/olaf/olaf/commons/prompts.py)

Pipeline and corpus definition 

In [3]:

spacy_model = spacy.load("en_core_web_sm")

corpus_path = os.path.join(os.getenv('DATA_PATH'), "demo.txt")


corpus = [
"Alice is 25 years old. Bob, her brother, is 30 years old.",
"Alex has a dog called Ouper. Claire's dog is Ouper's best friend.",
"Martine is 22 years old. Leo is 27. Leo has a cousin that is 22 years old. Martine has a cousin that is 27 years old.",
"Nicolas and Sarah are first cousins. Their grandmother, Louise, is 80.",
"Paul and Marie are married. Their son, Thomas, is 10."
]


pipeline = Pipeline(
	spacy_model=spacy_model,
	corpus=[doc for doc in spacy_model.pipe(corpus)]
    )



defining llmcomponent based on deepseek

In [4]:
deepseek_generator = DeepSeekGenerator()
llm_cterm_extraction = LLMTermExtraction(
	prompt_template=deepseek_prompt_concept_term_extraction,
	llm_generator=deepseek_generator
)
pipeline.add_pipeline_component(llm_cterm_extraction)


llm_concept_extraction = LLMBasedConceptExtraction(
	deepseek_prompt_concept_extraction, deepseek_generator)
pipeline.add_pipeline_component(llm_concept_extraction)

llm_term_extraction = LLMTermExtraction(
	prompt_template=deepseek_prompt_relation_term_extraction,
	llm_generator=deepseek_generator
)
pipeline.add_pipeline_component(llm_term_extraction)


llm_relation_extraction = LLMBasedRelationExtraction(
	deepseek_prompt_relation_extraction, deepseek_generator)
pipeline.add_pipeline_component(llm_relation_extraction)

[2025-03-09 05:09:36,435] [WARNING] [llm_based_relation_extraction] [_check_parameters] [No value given for concept_max_distance parameter, default will be set to 5.]


running pipeline and displaying Knowledge Rpresentation

In [5]:
pipeline.run()


kr_serialiser = KRJSONSerialiser()
kr_serialisation_path = os.path.join(
	os.getcwd(), "llm_deepseek_pipeline_kr.json")
kr_serialiser.serialise(kr=pipeline.kr, file_path=kr_serialisation_path)

kr_rdf_graph_path = os.path.join(
	os.getcwd(), "llm_deepseek_kr_rdf_graph.ttl")
pipeline.kr.rdf_graph.serialize(kr_rdf_graph_path, format="ttl")

print(f"Nb concepts: {len(pipeline.kr.concepts)}")
print(f"Nb relations: {len(pipeline.kr.relations)}")
print(f"Nb metarelations: {len(pipeline.kr.metarelations)}")
print(f"The KR object has been JSON serialised in : {kr_serialisation_path}")
print(f"The KR RDF graph has been serialised in : {kr_rdf_graph_path}")

Nb concepts: 9
Nb relations: 16
Nb metarelations: 0
The KR object has been JSON serialised in : /home/talibe/Bureau/Stage Insa/OLAF Research/olaf/demonstrators/llm_deepseek_pipeline_kr.json
The KR RDF graph has been serialised in : /home/talibe/Bureau/Stage Insa/OLAF Research/olaf/demonstrators/llm_deepseek_kr_rdf_graph.ttl


In [2]:
my_olaf_demo_serialiser = BaseOWLSerialiser("http://olaf_demo_results.org/")
my_olaf_demo_serialiser.build_graph(pipeline.kr)

NameError: name 'BaseOWLSerialiser' is not defined